In [12]:


import pandas as pd                          
from sklearn.model_selection import train_test_split  
from sklearn.preprocessing  import LabelEncoder      
from sklearn.ensemble       import RandomForestClassifier 
from sklearn.metrics        import accuracy_score, classification_report 
import warnings
warnings.filterwarnings('ignore')  

In [13]:
df = pd.read_csv('telecom_churn_mock_data.csv')

df.shape      
df.head()    

,CustomerID,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST2999,Male,0,Yes,Yes,71,No,No,No,No,...,No,No,No,No,One year,No,Electronic check,120.00,8756.02,No
1,CUST2998,Male,0,No,No,19,Yes,No,No,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,84.09,1642.44,No
2,CUST2997,Male,0,No,No,57,Yes,No,DSL,No,...,No,No,Yes,No,Month-to-month,No,Electronic check,82.28,4805.18,No
3,CUST2996,Female,0,No,Yes,34,Yes,No,DSL,No internet service,...,No internet service,No,No internet service,No,Month-to-month,Yes,Mailed check,120.00,3951.13,No
4,CUST2995,Female,0,No,Yes,72,Yes,No,Fiber optic,No,...,No,Yes,Yes,No,Month-to-month,No,Bank transfer (automatic),74.29,5206.46,No


In [14]:
# CustomerID is just a name/ID — it has no pattern to learn from
# So we drop (remove) it

df = df.drop('CustomerID', axis=1)  
print(df.shape)  

(2000, 20)


In [15]:


le = LabelEncoder()   # create the converter

for col in df.select_dtypes(include=['object']).columns:

    df[col] = le.fit_transform(df[col])   # convert it to numbers

print(df.head(3)) 

   Gender  SeniorCitizen  Partner  Dependents  Tenure  PhoneService  \
0       1              0        1           1      71             0   
1       1              0        0           0      19             1   
2       1              0        0           0      57             1   

   MultipleLines  InternetService  OnlineSecurity  OnlineBackup  \
0              0                2               0             0   
1              0                2               0             0   
2              0                0               0             2   

   DeviceProtection  TechSupport  StreamingTV  StreamingMovies  Contract  \
0                 0            0            0                0         1   
1                 0            0            0                0         0   
2                 0            0            2                0         0   

   PaperlessBilling  PaymentMethod  MonthlyCharges  TotalCharges  Churn  
0                 0              2          120.00       8756.02   

In [16]:
print(df.isnull().sum())

Gender              0
SeniorCitizen       0
Partner             0
Dependents          0
Tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [17]:
df.head()

,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,1,0,1,1,71,0,0,2,0,0,0,0,0,0,1,0,2,120.00,8756.02,0
1,1,0,0,0,19,1,0,2,0,0,0,0,0,0,0,1,2,84.09,1642.44,0
2,1,0,0,0,57,1,0,0,0,2,0,0,2,0,0,0,2,82.28,4805.18,0
3,0,0,0,1,34,1,0,0,1,1,1,0,1,0,0,1,3,120.00,3951.13,0
4,0,0,0,1,72,1,0,1,0,1,0,2,2,0,0,0,0,74.29,5206.46,0


In [18]:
# X = everything the model uses to LEARN (all columns except Churn)
# y = what we want the model to PREDICT (only the Churn column)

X = df.drop('Churn', axis=1)  
y = df['Churn']               # 1 output column (answer: 0=stayed, 1=left)

print("X shape:", X.shape)    
print("y shape:", y.shape)   

X shape: (2000, 19)
y shape: (2000,)


In [19]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       
    random_state=42     
)

print("Train rows:", X_train.shape[0])  
print("Test rows :", X_test.shape[0])    

Train rows: 1600
Test rows : 400


In [23]:
# Random Forest = 100 decision trees working together
model = RandomForestClassifier(
    n_estimators=100,      
  
          
    random_state=42       
)

model.fit(X_train, y_train)   

print("Model trained!")

Model trained!


In [ ]:
y_pred = model.predict(X_test)    

accuracy = accuracy_score(y_test, y_pred) 
print(f"Accuracy: {accuracy * 100:.2f}%")   # prints something like: Accuracy: 69.75%

print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))


Accuracy: 69.75%
              precision    recall  f1-score   support

      Stayed       0.72      0.94      0.81       278
     Churned       0.51      0.16      0.24       122

    accuracy                           0.70       400
   macro avg       0.61      0.55      0.53       400
weighted avg       0.65      0.70      0.64       400



In [ ]:

importance = pd.DataFrame({
    'Feature'   : X.columns,
    'Importance': model.feature_importances_
})

importance = importance.sort_values('Importance', ascending=False)
print(importance.head(5))  
# Result: TotalCharges, MonthlyCharges, Tenure are top 3

           Feature  Importance
18    TotalCharges    0.161303
17  MonthlyCharges    0.155376
4           Tenure    0.143683
16   PaymentMethod    0.057588
14        Contract    0.044673
